In [ ]:
# -------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------

PRODUCTION_DB_PATH = "../web/data/Track.db"
ALLOW_DB_WRITES    = True                           # Safety guard: must be True to write to Track.db
DB_PATH            = PRODUCTION_DB_PATH             # Scraper writes directly to Track.db
YEAR               = 2026
GENDER             = "Boys"                        # "Boys" or "Girls"
MEET_TYPE          = "Sectional"                    # "Sectional", "Regional", or "State"
SEC_TO_PROCESS     = list(range(1, 33))             # Sectionals 1-32
REG_TO_PROCESS     = list(range(1, 9))              # Regionals 1-8
INDEX_MONTHS       = [5, 6]                         # Search both months for robustness
IHSAA_FROM_YEAR    = 2026                           # Use IHSAA round pages for this year and newer
MIN_ROWS_EXPECTED  = 30                             # Warning threshold for suspiciously low parses

# -------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------

import importlib
import requests
import re
import gc
import logging
import time
import random
import shutil
import sqlite3
import sys
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin
from pathlib import Path

# Ensure project root is on sys.path so util imports work in notebook context.
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
repo_root = next((p for p in candidates if (p / "util" / "school_mappings.py").exists()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from util.school_mappings import team_mapping as shared_team_mapping
import util.db_util as db_util_module
import util.conversion_util as conversion_util_module

db_util_module = importlib.reload(db_util_module)
conversion_util_module = importlib.reload(conversion_util_module)
Database = db_util_module.Database
Conversion = conversion_util_module.Conversion

# -------------------------------------------------------------
# CONSTANTS
# -------------------------------------------------------------

BASE    = "https://in.milesplit.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

FIELD_EVENTS = {"High Jump", "Long Jump", "Triple Jump", "Shot Put", "Discus", "Pole Vault"}
RELAY_EVENTS = {"4 x 100 Relay", "4 x 400 Relay", "4 x 800 Relay"}
NO_MARK      = {"NT", "ND", "NH", "NM", "DNS", "DNF", "DQ", "SCR", "--"}
GRADES       = {"FR", "SO", "JR", "SR", "9", "10", "11", "12", "Freshman", "Sophomore", "Junior", "Senior"}

EVENT_ALIASES = {
    "100 meters": "100 Meters", "100 meter dash": "100 Meters", "100 meter run": "100 Meters",
    "200 meters": "200 Meters", "200 meter dash": "200 Meters", "200 meter run": "200 Meters",
    "400 meters": "400 Meters", "400 meter dash": "400 Meters", "400 meter run": "400 Meters",
    "800 meters": "800 Meters", "800 meter run": "800 Meters",
    "1600 meters": "1600 Meters", "1600 meter run": "1600 Meters", "mile run": "1600 Meters",
    "3200 meters": "3200 Meters", "3200 meter run": "3200 Meters", "2 mile run": "3200 Meters",
    "110 meter hurdles": "110 Hurdles", "110 hurdles": "110 Hurdles", "110m hurdles": "110 Hurdles",
    "100 meter hurdles": "100 Hurdles", "100 hurdles": "100 Hurdles", "100m hurdles": "100 Hurdles",
    "300 meter hurdles": "300 Hurdles", "300 hurdles": "300 Hurdles", "300m hurdles": "300 Hurdles",
    "4 x 100 meter relay": "4 x 100 Relay", "4 x 100 relay": "4 x 100 Relay", "4x100": "4 x 100 Relay", "4x100 meter relay" : "4 x 100 Relay",
    "4 x 400 meter relay": "4 x 400 Relay", "4 x 400 relay": "4 x 400 Relay", "4x400": "4 x 400 Relay", "4x400 meter relay": "4 x 400 Relay",
    "4 x 800 meter relay": "4 x 800 Relay", "4 x 800 relay": "4 x 800 Relay", "4x800": "4 x 800 Relay", "4x800 meter relay": "4 x 800 Relay",
    "high jump": "High Jump", "long jump": "Long Jump", "triple jump": "Triple Jump",
    "shot put": "Shot Put", "discus": "Discus", "discus throw": "Discus", "pole vault": "Pole Vault",
    "team scores": "Team Scores",   
}

# -------------------------------------------------------------
# SETUP
# -------------------------------------------------------------

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")
session = requests.Session()
session.headers.update(HEADERS)

if not ALLOW_DB_WRITES:
    raise RuntimeError("Set ALLOW_DB_WRITES = True to run scraper against Track.db")

DB_PATH = PRODUCTION_DB_PATH
prod_path = Path(DB_PATH).resolve()

backup_ans = input(
    "Do you want to create a backup of Track.db for rollback purposes? Type 'yes' to create, or press Enter to skip: "
).strip().lower()

if backup_ans == "yes":
    backup_candidates = sorted(prod_path.parent.glob("Track_backup_*.db"), key=lambda p: p.name)
    if backup_candidates:
        latest_backup = backup_candidates[-1]
        archive_path = prod_path.parent / "Track_archive.db"
        shutil.copy2(latest_backup, archive_path)
        print(f"Renamed old backup to: {archive_path.name}")

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = prod_path.parent / f"Track_backup_{ts}.db"
    shutil.copy2(prod_path, backup_path)
    print(f"Created backup: {backup_path.name}")
else:
    print("Skipping backup creation.")

    restore_ans = input(
        "Do you want to restore the most recent backup of Track.db? Type 'yes' to restore, or press Enter to skip: "
    ).strip().lower()
    if restore_ans == "yes":
        backup_candidates = sorted(prod_path.parent.glob("Track_backup_*.db"), key=lambda p: p.name)
        if backup_candidates:
            latest_backup = backup_candidates[-1]
            shutil.copy2(latest_backup, prod_path)
            print(f"Track.db restored from backup: {latest_backup.name}")
        else:
            print("No Track_backup_*.db file found. Restore aborted.")
    else:
        print("Keeping current Track.db (no restore).")

print(f"Active DB: {DB_PATH}")

db = Database(DB_PATH)
convert = Conversion()
warningDF = pd.DataFrame(columns=["warning", "id", "desc"])
runStats = []

# -------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------

def log_warning(warning, id_value, desc=""):
    if str(id_value) not in warningDF["id"].values:
        warningDF.loc[len(warningDF)] = {
            "warning": warning,
            "id": str(id_value),
            "desc": str(desc)
        }


def safe_get(url, retries=4, params=None):
    for attempt in range(retries + 1):
        time.sleep(1.5 + random.uniform(0, 1.5))
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt == retries:
                log_warning("Fetch failed", url, str(e))
                return None
            time.sleep(min(10, 2 ** attempt))


def normalize_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


def grade_normalization(grade):
    return {
        "Freshman": "FR", "Fr": "FR", "FR": "FR", "9": "FR",
        "Sophomore": "SO", "So": "SO", "SO": "SO", "10": "SO",
        "Junior": "JR", "Jr": "JR", "JR": "JR", "11": "JR",
        "Senior": "SR", "Sr": "SR", "SR": "SR", "12": "SR"
    }.get(str(grade).strip(), "Unknown")


def calculate_grad_year(grade_raw):
    offset = {"SR": 0, "JR": 1, "SO": 2, "FR": 3}.get(grade_normalization(grade_raw), -1)
    return YEAR + offset if offset != -1 else 9999


def team_mapping(team):
    print(team)
    mapped = shared_team_mapping(team)
    mapped = re.sub(r"\b(?:HS|High School)\b", "", str(mapped), flags=re.IGNORECASE)
    mapped = re.sub(r"\s+", " ", mapped).strip(" -")
    print(f"Mapped team: {mapped}")
    return mapped


_school_lookup_cache = None


def _normalize_school_name(name):
    if not name:
        return ""

    normalized = str(name).strip()
    normalized = normalized.replace("&", " and ")
    normalized = re.sub(r"^FW\s+", "Fort Wayne ", normalized, flags=re.IGNORECASE)
    normalized = re.sub(r"^Ft\.?\s+", "Fort ", normalized, flags=re.IGNORECASE)
    normalized = re.sub(r"\s*\([^)]*\)", " ", normalized)
    normalized = re.sub(r"[^a-zA-Z0-9]+", " ", normalized).lower().strip()

    drop_tokens = {
        "high", "school", "community", "institute", "academy", "junior", "senior", "jr", "sr", "jr sr",
    }
    tokens = [t for t in normalized.split() if t not in drop_tokens]
    return " ".join(tokens)


def _load_school_lookup_cache():
    global _school_lookup_cache
    if _school_lookup_cache is not None:
        return _school_lookup_cache

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT school_id, school_name, team_name FROM school")
    rows = cur.fetchall()
    conn.close()

    entries = []
    by_normalized = {}
    for school_id, school_name, team_name in rows:
        for n in [_normalize_school_name(school_name), _normalize_school_name(team_name)]:
            if not n:
                continue
            token_set = set(n.split())
            entry = {
                "school_id": school_id,
                "school_name": school_name,
                "normalized": n,
                "tokens": token_set,
            }
            entries.append(entry)
            by_normalized.setdefault(n, []).append(school_id)

    _school_lookup_cache = {"entries": entries, "by_normalized": by_normalized}
    return _school_lookup_cache


def _resolve_school_id(team_name):
    school_id = db.get_school_id(team_name)
    if school_id is not None:
        return school_id

    norm_input = _normalize_school_name(team_name)
    if not norm_input:
        return None

    cache = _load_school_lookup_cache()
    by_normalized = cache["by_normalized"]
    if norm_input in by_normalized and len(by_normalized[norm_input]) == 1:
        return by_normalized[norm_input][0]

    input_tokens = set(norm_input.split())
    scored = []
    for entry in cache["entries"]:
        db_tokens = entry["tokens"]
        overlap = len(input_tokens.intersection(db_tokens))
        if overlap == 0:
            continue
        score = (2.0 * overlap) / (len(input_tokens) + len(db_tokens))
        if score >= 1.0:
            scored.append((score, overlap, entry["school_id"]))

    if not scored:
        return None

    scored.sort(reverse=True)
    best_score, best_overlap, best_id = scored[0]
    top_ids = [item[2] for item in scored if item[0] == best_score and item[1] == best_overlap]
    return best_id if len(set(top_ids)) == 1 else None


def normalize_event(raw):
    raw = re.sub(r"^(boys'?|girls'?)(\s+(boys|girls))?\s*[-\s]*", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"^(varsity|jv|junior varsity|freshman|frosh|sophomore|results)\s+", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"\s+high school\s+(boys|girls)\s*$", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s+(boys|girls)\s*$", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s+(Finals?|Preliminaries?|Prelims?)$", "", raw.strip(), flags=re.IGNORECASE)
    return EVENT_ALIASES.get(raw.lower().strip())


def _matches_expected_meet_type(text, expected_meet_type):
    if not expected_meet_type:
        return True
    t = normalize_text(text)
    if not t:
        return True

    mt = str(expected_meet_type).strip().lower()
    has_sectional = "sectional" in t
    has_regional = "regional" in t
    has_state = "state" in t

    if mt == "sectional":
        return not has_regional and not has_state
    if mt == "regional":
        return not has_sectional and not has_state
    if mt == "state":
        return not has_sectional and not has_regional
    return True


def _matches_expected_gender(text, expected_gender):
    if not expected_gender:
        return True
    t = normalize_text(text)
    if not t:
        return True

    wants_boys = str(expected_gender).strip().lower() == "boys"
    has_boys = "boys" in t
    has_girls = "girls" in t

    if has_boys and not has_girls:
        return wants_boys
    if has_girls and not has_boys:
        return not wants_boys
    return True


def _season_slug(championship_year):
    start_year = int(championship_year) - 1
    return f"{start_year}-{int(championship_year) % 100:02d}"


def _ihsaa_round_slug(meet_type):
    mt = (meet_type or "").strip().lower()
    if mt == "sectional":
        return "sectionals"
    if mt == "regional":
        return "regionals"
    return "state-finals"


def _ihsaa_tournament_url(year, gender, meet_type):
    gender_slug = "boys" if str(gender).strip().lower() == "boys" else "girls"
    season_slug = _season_slug(int(year))
    round_slug = _ihsaa_round_slug(meet_type)
    return f"https://www.ihsaa.org/sports/{gender_slug}/track-field/{season_slug}-tournament?round={round_slug}"


def _extract_ihsaa_num_host(text, default_num=None):
    s = re.sub(r"\s+", " ", text or "").strip()
    m = re.search(r"(?:^|\b)(\d{1,2})\.\s*([^|]+?)\s*\(", s)
    if m:
        return int(m.group(1)), m.group(2).strip()

    host = None
    m_host = re.search(r"\b([A-Z][A-Za-z'().\-]+(?:\s+[A-Z][A-Za-z'().\-]+){0,5})\s*\(", s)
    if m_host:
        host = m_host.group(1).strip()

    if default_num is not None:
        return int(default_num), host
    return None, host


def _extract_result_ids_from_meet_page(meet_url):
    result_ids = set()
    visited = set()
    queue = [meet_url]

    def _collect_ids_from_text(value):
        if not value:
            return
        for m in re.finditer(r"/results/(\d+)", str(value)):
            result_ids.add(m.group(1))

    def _should_follow_link(href):
        h = str(href or "").strip().lower()
        if not h:
            return False
        if h.startswith("javascript:") or h.startswith("mailto:") or h.startswith("tel:"):
            return False
        full = h if h.startswith("http") else urljoin(BASE, h)
        # Follow any plausible MileSplit meet/results page regardless of label text.
        if "in.milesplit.com" not in full:
            return False
        return ("/meets/" in full) or ("/results" in full)

    # Search the meet page and one level of linked result pages.
    max_hops = 2
    hop = 0
    while queue and hop < max_hops:
        next_queue = []
        for page_url in queue:
            if not page_url or page_url in visited:
                continue
            visited.add(page_url)

            _collect_ids_from_text(page_url)

            r = safe_get(page_url)
            if r is None:
                continue

            soup = BeautifulSoup(r.text, "html.parser")

            follow_count = 0
            for a in soup.find_all("a", href=True):
                href = (a.get("href") or "").strip()
                _collect_ids_from_text(href)

                if _should_follow_link(href):
                    follow_url = href if href.startswith("http") else urljoin(BASE, href)
                    if follow_url not in visited:
                        next_queue.append(follow_url)
                        follow_count += 1
                        if follow_count >= 75:
                            break

            for opt in soup.find_all("option"):
                val = (opt.get("value") or "").strip()
                _collect_ids_from_text(val)

            # Some MileSplit pages stash links in inline scripts.
            for script in soup.find_all("script"):
                _collect_ids_from_text(script.get_text(" ", strip=True))

        queue = next_queue
        hop += 1

    return sorted(result_ids)


def _extract_meet_id(meet_url):
    m = re.search(r"/meets/(\d+)", meet_url or "")
    return m.group(1) if m else None


def _is_target_meet(anchor_text, meet_type, meet_num, gender):
    t = normalize_text(anchor_text)
    if "ihsaa" not in t:
        return False
    if gender.lower() not in t:
        return False

    mt = meet_type.lower()
    if mt == "state":
        return ("state" in t) and ("sectional" not in t) and ("regional" not in t)

    if mt not in t:
        return False

    return bool(re.search(rf"\b{re.escape(str(meet_num))}\b", t))


def get_ihsaa_meet_links(meet_type, gender, year):
    url = _ihsaa_tournament_url(year, gender, meet_type)
    r = safe_get(url)
    if r is None:
        return {}

    soup = BeautifulSoup(r.text, "html.parser")
    links = {}
    default_num = 1 if (meet_type or "").strip().lower() == "state" else None

    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        if "in.milesplit.com" not in href or "/results" not in href:
            continue

        anchor_label = a.get_text(" ", strip=True)
        if "result" not in normalize_text(anchor_label):
            continue

        meet_url = href if href.startswith("http") else urljoin(BASE, href)
        row_text = a.parent.get_text(" ", strip=True) if a.parent is not None else anchor_label

        if not _matches_expected_meet_type(row_text + " " + meet_url, meet_type):
            continue
        if not _matches_expected_gender(row_text + " " + meet_url, gender):
            continue

        meet_num, host = _extract_ihsaa_num_host(row_text, default_num=default_num)
        if meet_num is None:
            continue

        links.setdefault(meet_num, []).append({
            "anchor_text": row_text,
            "meet_url": meet_url,
            "host": host,
            "source": "IHSAA"
        })

    return links


def get_meet_formatted_targets(meet_type, meet_num):
    # Prefer IHSAA round page lookup for modern years.
    if YEAR >= IHSAA_FROM_YEAR:
        ihsaa_map = get_ihsaa_meet_links(meet_type, GENDER, YEAR)
        candidates = ihsaa_map.get(int(meet_num), [])
        if candidates:
            c = candidates[0]
            result_ids = _extract_result_ids_from_meet_page(c["meet_url"])
            return {
                "source": c.get("source", "IHSAA"),
                "anchor_text": c.get("anchor_text", ""),
                "meet_url": c.get("meet_url"),
                "host": c.get("host"),
                "meet_id": _extract_meet_id(c.get("meet_url")),
                "results_ids": result_ids,
            }

    # Fallback: monthly MileSplit search.
    selected = None
    for month in INDEX_MONTHS:
        page = 1
        max_pages = 5

        while page <= max_pages:
            index_url = f"{BASE}/results?month={month}&year={YEAR}&level=hs&page={page}"
            r = safe_get(index_url)
            if r is None:
                break

            soup = BeautifulSoup(r.text, "html.parser")
            for a in soup.find_all("a", href=True):
                text = a.get_text(" ", strip=True)
                href = a.get("href", "").strip()
                if not text or "/meets/" not in href:
                    continue
                if not _is_target_meet(text, meet_type, meet_num, GENDER):
                    continue

                meet_url = href if href.startswith("http") else urljoin(BASE, href)
                selected = {"anchor_text": text, "meet_url": meet_url, "source": "MileSplit"}
                break

            if selected is not None:
                break

            if not soup.find("a", string=re.compile(r"next", re.IGNORECASE)):
                break
            page += 1

        if selected is not None:
            break

    if selected is None:
        return {
            "source": "MileSplit",
            "anchor_text": "",
            "meet_url": None,
            "host": None,
            "meet_id": None,
            "results_ids": [],
        }

    anchor_text = selected["anchor_text"]
    meet_url = selected["meet_url"]
    host = anchor_text.split("-")[-1].strip() if "-" in anchor_text else anchor_text.strip()

    return {
        "source": selected["source"],
        "anchor_text": anchor_text,
        "meet_url": meet_url,
        "host": host,
        "meet_id": _extract_meet_id(meet_url),
        "results_ids": _extract_result_ids_from_meet_page(meet_url),
    }


def fetch_formatted_api_records(meet_id, results_id, label=""):
    api_url = f"{BASE}/api/v1/meets/{meet_id}/performances"
    fields = (
        "id,meetId,eventName,roundName,place,mark,statusCode,"
        "firstName,lastName,teamName,gradYear,units,millimeters"
    )
    r = safe_get(api_url, params={"resultsId": str(results_id), "fields": fields})
    if r is None:
        log_warning("Formatted API fetch failed", label)
        return []

    try:
        payload = r.json()
    except Exception as e:
        log_warning("Formatted API JSON parse error", label, str(e))
        return []

    data = payload.get("data") if isinstance(payload, dict) else None
    if not isinstance(data, list):
        log_warning("Formatted API returned no data list", label)
        return []

    records = []
    for row in data:
        raw_event = (row.get("eventName") or "").strip()
        event_name = normalize_event(raw_event)
        if event_name is None:
            log_warning("Formatted API unknown event skipped", raw_event, label)
            print(f"  Warning: unknown event skipped: '{raw_event}' (label: {label})")
            continue

        round_str = (row.get("roundName") or "").strip().lower()
        event_type = "Prelim" if "prelim" in round_str else "Final"
        place = str(row.get("place") or "").strip()
        if not place:
            continue

        status = (row.get("statusCode") or "").strip()
        mark = (row.get("mark") or "").strip()
        if not mark and status:
            mark = status
        if not mark and event_name not in FIELD_EVENTS:
            try:
                seconds = int(row.get("units")) / 1000.0
                mins = int(seconds // 60)
                mark = f"{mins}:{seconds - mins * 60:05.2f}" if mins > 0 else f"{seconds:.2f}"
            except Exception:
                mark = ""

        if not mark or mark.upper() in NO_MARK:
            continue

        if event_name in FIELD_EVENTS:
            mark = mark.rstrip("\".")

        first = (row.get("firstName") or "").strip()
        last = (row.get("lastName") or "").strip()
        name = f"{first} {last}".strip()
        team = (row.get("teamName") or "").strip()

        grade = ""
        try:
            offset = int(row.get("gradYear") or 9999) - YEAR
            grade = {0: "SR", 1: "JR", 2: "SO", 3: "FR"}.get(offset, "")
        except Exception:
            pass

        if event_name in RELAY_EVENTS:
            if not team:
                continue
            records.append({
                "event_name": event_name,
                "event_type": event_type,
                "place": place,
                "name": "",
                "grade": "",
                "team": team,
                "mark": mark,
                "relay_athletes": [],
            })
        else:
            if not name or not team:
                continue
            records.append({
                "event_name": event_name,
                "event_type": event_type,
                "place": place,
                "name": name,
                "grade": grade,
                "team": team,
                "mark": mark,
                "relay_athletes": [],
            })

    return records


_athlete_cache = {}


def process_athlete(full_name, school_id, grad_year):
    parts = full_name.strip().split()
    if not parts:
        return None

    first = parts[0].upper()
    last = " ".join(parts[1:]).upper() if len(parts) > 1 else ""
    key = (first, last, school_id, grad_year)

    if key in _athlete_cache:
        return _athlete_cache[key]

    athlete_id = db.get_athlete_id(first, last, school_id, grad_year)
    if athlete_id is None:
        athlete_id = db.get_athlete_id_by_name_school(first, last, school_id)
        if athlete_id is not None:
            existing_grad = db.get_athlete_grad_year(athlete_id)
            if grad_year != 9999 and existing_grad != grad_year:
                db.update_athlete_grad_year(athlete_id, grad_year, commit=False)
                log_warning("Grad year updated", f"{first} {last}", f"school_id={school_id} old={existing_grad} new={grad_year}")
        else:
            athlete_id = db.insert_athlete(school_id, first, last, GENDER, grad_year, commit=False)
            log_warning("Athlete created", f"{first} {last}", f"school_id={school_id} grad={grad_year}")

    _athlete_cache[key] = athlete_id
    return athlete_id

# -------------------------------------------------------------
# DETERMINE MEETS TO PROCESS
# -------------------------------------------------------------

if MEET_TYPE == "Sectional":
    meets_to_process = SEC_TO_PROCESS
elif MEET_TYPE == "Regional":
    meets_to_process = REG_TO_PROCESS
else:
    meets_to_process = [1]

# -------------------------------------------------------------
# MAIN (FORMATTED API ONLY)
# -------------------------------------------------------------

print("MileSplit formatted scraper started.")
start_time = datetime.now()
print(f"Start time: {start_time.strftime('%H:%M:%S')}")

for meet_num in meets_to_process:
    label = f"{YEAR} {GENDER} {MEET_TYPE}" + (f" {meet_num}" if MEET_TYPE != "State" else "")
    print(f"Processing {label}...")

    meet_id = db.get_meet_id(MEET_TYPE, meet_num, YEAR, GENDER)
    if meet_id is not None:
        print(f"  Meet already exists in meet table (meet_id={meet_id}); skipping scrape.")
        print("  Rows read in: 0")
        runStats.append({
            "label": label,
            "source": "",
            "anchor_text": "",
            "meet_url": "",
            "raw_url": "",
            "raw_option_found": 0,
            "parser_format": "",
            "parsed_rows": 0,
            "athlete_rows_inserted": 0,
            "relay_rows_inserted": 0,
            "status": "meet_exists_skipped",
            "note": f"Meet already exists in meet table (meet_id={meet_id})"
        })
        continue

    meet_info = get_meet_formatted_targets(MEET_TYPE, meet_num)
    meet_url = meet_info.get("meet_url")
    meet_host = meet_info.get("host")
    meet_id_api = meet_info.get("meet_id")
    results_ids = meet_info.get("results_ids", [])

    print(f"  Meet URL: {meet_url or 'N/A'}")
    if results_ids:
        print(f"  Results IDs found: {', '.join(results_ids)}")

    meet_stat = {
        "label": label,
        "source": meet_info.get("source", ""),
        "anchor_text": meet_info.get("anchor_text", ""),
        "meet_url": meet_url or "",
        "raw_url": "",  # kept for output compatibility; stores chosen results URL
        "raw_option_found": int(bool(results_ids)),
        "parser_format": "formatted_api",
        "parsed_rows": 0,
        "athlete_rows_inserted": 0,
        "relay_rows_inserted": 0,
        "status": "ok",
        "note": ""
    }

    if not meet_url or not meet_id_api:
        log_warning("Meet URL not found", label)
        print("  Rows read in: 0")
        meet_stat["status"] = "meet_not_found"
        meet_stat["note"] = "No matching meet link"
        runStats.append(meet_stat)
        continue

    if not results_ids:
        log_warning("No resultsId found on meet page", meet_url)
        print("  Rows read in: 0")
        meet_stat["status"] = "formatted_missing_results_id"
        meet_stat["note"] = "No resultsId candidates found"
        runStats.append(meet_stat)
        continue

    best_records = []
    best_results_id = None
    for rid in results_ids:
        recs = fetch_formatted_api_records(meet_id_api, rid, label=label)
        if len(recs) > len(best_records):
            best_records = recs
            best_results_id = rid

    records = best_records
    parsed_count = len(records)
    meet_stat["parsed_rows"] = parsed_count
    meet_stat["raw_url"] = f"{BASE}/meets/{meet_id_api}/results/{best_results_id}" if best_results_id else ""

    if best_results_id:
        print(f"  Selected results URL: {BASE}/meets/{meet_id_api}/results/{best_results_id}")

    if parsed_count < MIN_ROWS_EXPECTED:
        meet_stat["note"] = f"Parsed rows below threshold ({parsed_count} < {MIN_ROWS_EXPECTED})"

    if parsed_count == 0:
        meet_stat["status"] = "formatted_no_rows"
        print("  Rows read in: 0")
        runStats.append(meet_stat)
        continue

    print(f"  Parsed {parsed_count} rows from formatted API.")

    db.insert_meet(meet_host or label, MEET_TYPE, meet_num, YEAR, GENDER, commit=False)
    meet_id = db.get_meet_id(MEET_TYPE, meet_num, YEAR, GENDER)

    for rec in records:
        event_name = rec["event_name"]
        event_type = rec["event_type"]
        place = rec["place"]
        mark = rec["mark"]
        team = team_mapping(rec["team"])
        grade = rec["grade"]

        if not team:
            log_warning("Empty team name - skipping", rec.get("name", ""), label)
            continue

        is_relay = event_name in RELAY_EVENTS

        try:
            mark2 = convert.distance_to_inches(mark) if event_name in FIELD_EVENTS else convert.time_to_seconds(mark)
        except Exception:
            mark2 = None

        school_id = _resolve_school_id(team)
        if school_id is None:
            log_warning("School not found", team, label)
            print(f"    Warning: School '{team}' not found in database; skipping record.")
            continue

        if is_relay:
            if db.get_relay_result(school_id, meet_id, event_name) is None:
                db.insert_relay_result(school_id, meet_id, event_name, mark, mark2, place, "", commit=False)
                meet_stat["relay_rows_inserted"] += 1
        else:
            if not grade:
                parts = rec["name"].strip().split()
                first = parts[0].upper() if parts else ""
                last = " ".join(parts[1:]).upper() if len(parts) > 1 else ""
                existing = db.get_athlete_id_by_name_school(first, last, school_id) if first else None
                if existing is not None:
                    existing_grad = db.get_athlete_grad_year(existing)
                    offset = existing_grad - YEAR
                    grade = {0: "SR", 1: "JR", 2: "SO", 3: "FR"}.get(offset, "")

            grad_year = calculate_grad_year(grade) if grade else 9999
            athlete_id = process_athlete(rec["name"], school_id, grad_year)
            if athlete_id is None:
                continue

            norm_grade = grade_normalization(grade) if grade else "Unknown"
            if db.get_athlete_result(athlete_id, meet_id, event_name, event_type) is None:
                db.insert_athlete_result(athlete_id, meet_id, event_name, event_type, norm_grade, mark, mark2, place, commit=False)
                meet_stat["athlete_rows_inserted"] += 1

    runStats.append(meet_stat)
    gc.collect()

db.do_commit()

end_time = datetime.now()
print(f"End time:     {end_time.strftime('%H:%M:%S')}")
print(f"Elapsed time: {end_time - start_time}")
total_rows_read = sum(int(s.get("parsed_rows", 0) or 0) for s in runStats)
print(f"Total rows read in: {total_rows_read}")

warnings_file = f"{YEAR} {GENDER} {MEET_TYPE} MileSplit Warnings.csv"
warningDF.to_csv(warnings_file, index=False)
print(f"Warnings written to: {warnings_file}")

stats_file = f"{YEAR} {GENDER} {MEET_TYPE} MileSplit Run Summary.csv"
pd.DataFrame(runStats).to_csv(stats_file, index=False)
print(f"Run summary written to: {stats_file}")
print("Done!")

Skipping backup creation.
Keeping current Track.db (no restore).
Active DB: ../web/data/Track.db
MileSplit formatted scraper started.
Start time: 11:52:33
Processing 2026 Boys Sectional 1...
  Meet already exists in meet table (meet_id=3257); skipping scrape.
  Rows read in: 0
Processing 2026 Boys Sectional 2...
  Meet already exists in meet table (meet_id=3258); skipping scrape.
  Rows read in: 0
Processing 2026 Boys Sectional 3...
  Meet already exists in meet table (meet_id=3259); skipping scrape.
  Rows read in: 0
Processing 2026 Boys Sectional 4...
  Meet already exists in meet table (meet_id=3260); skipping scrape.
  Rows read in: 0
Processing 2026 Boys Sectional 5...
  Meet already exists in meet table (meet_id=3261); skipping scrape.
  Rows read in: 0
Processing 2026 Boys Sectional 6...
  Meet already exists in meet table (meet_id=3262); skipping scrape.
  Rows read in: 0
Processing 2026 Boys Sectional 7...
  Meet already exists in meet table (meet_id=3263); skipping scrape.
  